In [1]:
import ultraplot as uplt
# import matplotlib.pyplot as plt
import numpy as np

In [231]:
import warnings

import healpy as hp
import numpy as np
import numpy.typing as npt
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.time import Time

from rubin_scheduler.utils import DEFAULT_NSIDE, _hpid2_ra_dec

from rubin_scheduler.scheduler.utils.sky_area import CurrentAreaMap
from rubin_scheduler.scheduler.utils.footprints import (
    Footprint,
    StepSlopes,
    Footprints,
)

# survey start from commit 5e34aa5102d3d44fc03c2a74934338fc2ba2d1b9
# on https://github.com/lsst-sims/sims_featureScheduler_runs5.3
SURVEY_START_MJD = Time("2026-06-15T12:00:00").mjd

YEAR = 365.25

def _is_in_ra_range(ra, low, high):
    _low = low % (2.0 * np.pi)
    _high = high % (2.0 * np.pi)
    if _low <= _high:
        return (ra >= _low) & (ra <= _high)
    else:
        return (ra >= _low) | (ra <= _high)


def slice_quad_galactic_cut(
    target_map, wfd_indx=None, ra_range=None, swap_phase=True, desi_y1_only=False,
    ngc=True, sgc=True
):
    """
    Helper function for generating rolling footprints

    Parameters
    ----------
    target_map : dict of HEALpix maps
        The final desired footprint as HEALpix maps. Keys are band names
    wfd_indx : array of ints
        The indices of target_map that should be used for rolling.
        If None, assumes the rolling area should be where target_map['r'] == 1.
    ra_range : tuple of floats, optional
        If not None, then the indices are restricted to the given RA range
        in radians.
    """
    nside = hp.npix2nside(target_map["r"].size)
    ra, dec = _hpid2_ra_dec(nside, np.arange(hp.nside2npix(nside)))

    coord = SkyCoord(ra=ra * u.rad, dec=dec * u.rad)
    _, gal_lat = coord.galactic.l.deg, coord.galactic.b.deg

    indx_north = np.intersect1d(np.where(gal_lat >= 0)[0], wfd_indx)
    indx_south = np.intersect1d(np.where(gal_lat < 0)[0], wfd_indx)

    splits_north = slice_wfd_area_desi_north(target_map, wfd_indx=indx_north, desi_y1_only=desi_y1_only)
    splits_south = slice_wfd_area_desi_south(target_map, wfd_indx=indx_south, desi_y1_only=desi_y1_only)

    slice_indx = [[], []]
    if swap_phase:
        indx0 = 1
        indx1 = 0
    else:
        indx0 = 0
        indx1 = 1

    if desi_y1_only:
        if sgc:
            slice_indx[indx0].extend(
                indx_south[splits_south[0] : splits_south[1]].tolist()
            )
        if ngc:
            slice_indx[indx0].extend(
                indx_north[splits_north[0] : splits_north[1]].tolist()
            )
    else:
        if sgc:
            slice_indx[indx0].extend(
                indx_south[splits_south[0] : splits_south[1]].tolist()
                + indx_south[splits_south[2] : splits_south[3]].tolist()
            )
            slice_indx[indx1].extend(
                indx_south[splits_south[1] : splits_south[2]].tolist()
                + indx_south[splits_south[3] : splits_south[4]].tolist()
            )
        if ngc:
            slice_indx[indx0].extend(
                indx_north[splits_north[0] : splits_north[1]].tolist()
                + indx_north[splits_north[2] : splits_north[3]].tolist()
            )
            slice_indx[indx1].extend(
                indx_north[splits_north[1] : splits_north[2]].tolist()
                + indx_north[splits_north[3] : splits_north[4]].tolist()
            )

    if ra_range is not None:
        ra_indx = np.where(_is_in_ra_range(ra, *ra_range))[0]
        for j in range(2):
            slice_indx[j] = np.intersect1d(ra_indx, slice_indx[j])

    return slice_indx


def slice_wfd_area_desi_south(target_map, wfd_indx=None, desi_y1_only=False):

    nside = hp.npix2nside(target_map["r"].size)
    _, dec = _hpid2_ra_dec(nside, np.arange(hp.nside2npix(nside)))

    wfd = target_map["r"] * 0
    if wfd_indx is None:
        wfd_indices = np.where(target_map["r"] == 1)[0]
    else:
        wfd_indices = wfd_indx
    wfd[wfd_indices] = 1

    # the second stripe in the north needs to cover DESI SGC stripes 4, 5, 6, 7
    # dec min and max of second strip
    if desi_y1_only:
        dec_max = (-8.875 + 1.7) / 180.0 * np.pi
    else:
        dec_max = (-3.375 + 1.7) / 180.0 * np.pi
    dec_min = (-15.375 - 1.7) / 180.0 * np.pi
    msk = (dec[wfd_indices] <= dec_max) & (dec[wfd_indices] >= dec_min)
    start_ind = int(np.min(np.where(msk)[0]))
    end_ind = int(np.max(np.where(msk)[0])) + 1
    split_wfd_indices = [0, start_ind, end_ind]

    # rest of the cuts move to make area equal
    nslice = wfd_indices.shape[0] // 4
    split_wfd_indices.append(split_wfd_indices[-1] + 2*nslice - split_wfd_indices[1])
    split_wfd_indices.append(wfd_indices.shape[0])

    afracs = []
    for i in range(4):
        afracs.append(np.sum(wfd[wfd_indx[split_wfd_indices[i]:split_wfd_indices[i+1]]]) / np.sum(wfd))
        print("south:", i, np.sum(wfd[wfd_indx[split_wfd_indices[i]:split_wfd_indices[i+1]]]) / np.sum(wfd))

    print("south total and halves:", np.sum(afracs), afracs[0] + afracs[2], afracs[1] + afracs[3])

    if desi_y1_only:
        return split_wfd_indices[1:3]
    else:
        return split_wfd_indices


def slice_wfd_area_desi_north(target_map, wfd_indx=None, desi_y1_only=False):

    nside = hp.npix2nside(target_map["r"].size)
    _, dec = _hpid2_ra_dec(nside, np.arange(hp.nside2npix(nside)))

    wfd = target_map["r"] * 0
    if wfd_indx is None:
        wfd_indices = np.where(target_map["r"] == 1)[0]
    else:
        wfd_indices = wfd_indx
    wfd[wfd_indices] = 1

    # the second stripe in the north needs to cover DESI NGC stripes 7, 8
    # dec min and max of second strip
    if desi_y1_only:
        dec_max = (-1.625 + 1.7) / 180.0 * np.pi
    else:
        dec_max = (3.875 + 1.7) / 180.0 * np.pi
    dec_min = (-8.125 - 1.7) / 180.0 * np.pi
    msk = (dec[wfd_indices] <= dec_max) & (dec[wfd_indices] >= dec_min)
    start_ind = int(np.min(np.where(msk)[0]))
    end_ind = int(np.max(np.where(msk)[0])) + 1
    split_wfd_indices = [0, start_ind, end_ind]

    # rest of the cuts move to make area equal
    nslice = wfd_indices.shape[0] // 4
    split_wfd_indices.append(split_wfd_indices[-1] + 2*nslice - split_wfd_indices[1])
    split_wfd_indices.append(wfd_indices.shape[0])

    afracs = []
    for i in range(4):
        afracs.append(np.sum(wfd[wfd_indx[split_wfd_indices[i]:split_wfd_indices[i+1]]]) / np.sum(wfd))
        print("north:", i, np.sum(wfd[wfd_indx[split_wfd_indices[i]:split_wfd_indices[i+1]]]) / np.sum(wfd))

    print("north total and halves:", np.sum(afracs), afracs[0] + afracs[2], afracs[1] + afracs[3])

    if desi_y1_only:
        return split_wfd_indices[1:3]
    else:
        return split_wfd_indices


# default function from commit e6d604718bcf3253434c8ddfee0250d9d09b5c8e
# on rubin_scheduler
def make_rolling_footprints(
    *,
    mjd_start,
    fp_hp=None,
    nside=DEFAULT_NSIDE,
    wfd_indx=None,
):
    """
    Generate rolling footprints

    Parameters
    ----------
    fp_hp : dict-like
        A dict with band name keys and HEALpix map values.
        Default None will load CurrentAreaMap. Assumes
        WFD is where r-band is 1.
    mjd_start : `float`
        The starting date of the survey.
    wfd_indx : array of ints
        The indices of the HEALpix map that are to be included in the rolling.

    Returns
    -------
    Footprints object
    """

    from rubin_scheduler.site_models import Almanac

    # Use the Almanac to find the position of the sun at the start of survey
    almanac = Almanac(mjd_start=mjd_start)
    sun_moon_info = almanac.get_sun_moon_positions(mjd_start)
    sun_ra_start = sun_moon_info["sun_RA"].copy()

    if fp_hp is None:
        sky = CurrentAreaMap(nside=nside)
        footprints, labels = sky.return_maps()
        fp_hp = {}
        for key in footprints.dtype.names:
            fp_hp[key] = footprints[key]

    hp_footprints = fp_hp

    nslice = 2
    scale = 0.9
    D = 1.0 - scale
    U = nslice - D * (nslice - 1)

    dvals = {
        0: "0",
        1: "1",
        D: "D",
        U: "U",
    }
    inv_dvals = {v: k for k, v in dvals.items()}

    # "1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1",

    str_all_slopes = [
        # original rolling with flip fixed
        # "1 1 U D 1 U D 1 U D 1 1 1 1 1 1 1",
        # "1 1 D U 1 D U 1 D U 1 1 1 1 1 1 1",
        # "1 1 U D 1 U D 1 U D 1 1 1 1 1 1 1",
        # "1 1 D U 1 D U 1 D U 1 1 1 1 1 1 1",
        # "1 1 1 U D 1 U D 1 U D 1 1 1 1 1 1",
        # "1 1 1 D U 1 D U 1 D U 1 1 1 1 1 1",
        # "1 1 1 U D 1 U D 1 U D 1 1 1 1 1 1",
        # "1 1 1 D U 1 D U 1 D U 1 1 1 1 1 1",

        # delay by 0.25 years
        "1 1 1 U D 1 U D 1 U D 1 1 1 1 1 1",
        "1 1 1 D U 1 D U 1 D U 1 1 1 1 1 1",
        "1 1 U D 1 U D 1 U D 1 1 1 1 1 1 1",
        "1 1 D U 1 D U 1 D U 1 1 1 1 1 1 1",
        "1 1 1 U D 1 U D 1 U D 1 1 1 1 1 1",
        "1 1 1 D U 1 D U 1 D U 1 1 1 1 1 1",
        "1 1 1 U D 1 U D 1 U D 1 1 1 1 1 1",
        "1 1 1 D U 1 D U 1 D U 1 1 1 1 1 1",

    ]
    slice_names = [
        "a", "b", "a", "b", "a", "b", "a", "b", "a", "b"
    ]
    slice_quads = [
        270, 270, 360, 360, 90, 90, 180, 180
    ]
    slice_split_ind = [
        0, 1, 0, 1, 0, 1, 0, 1
    ]

    all_slopes = []
    for sstr in str_all_slopes:
        all_slopes.append(
            [inv_dvals[s] for s in sstr.split(" ")]
        )

    for i, s in enumerate(all_slopes):
        sname = "slice " + slice_names[i] + " for quad %3d"  % slice_quads[i]
        print(sname + ": " + " ".join([dvals[x] for x in s]))

    fp_non_wfd = Footprint(mjd_start, sun_ra_start=sun_ra_start, nside=nside)
    rolling_footprints = []
    for i in range(len(all_slopes)):
        step_func = StepSlopes(rise=np.array(all_slopes[i], dtype=float))
        rolling_footprints.append(
            Footprint(
                mjd_start,
                sun_ra_start=sun_ra_start,
                step_func=step_func,
                nside=nside,
            )
        )

    # desi Y1 stripes uniform footprint
    #order is 270, 360, 90, 180
    desi_extra_slopes = [
        # "0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0",
        # "0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0",
        # "0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0",
        # "0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0",
        "0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0", # 270
        "0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0",
        "0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0",
        "0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0",
    ]
    desi_extra_ngc = [False, False, True, False]
    desi_extra_sgc = [True, True, False, True]
    desi_extra_quads = [270, 360, 90, 180]
    des_extra_ra_range = [
        (sun_ra_start + 1.5 * np.pi, sun_ra_start + 2.0 * np.pi),
        (sun_ra_start, sun_ra_start + 0.5 * np.pi),
        (sun_ra_start + 0.5 * np.pi, sun_ra_start + 1.0 * np.pi),
        (sun_ra_start + 1.0 * np.pi, sun_ra_start + 1.5 * np.pi)
    ]
    desi_extra_slopes = [
            [inv_dvals[s] for s in desi_extra_slopes[0].split(" ")],
            [inv_dvals[s] for s in desi_extra_slopes[1].split(" ")],
            [inv_dvals[s] for s in desi_extra_slopes[2].split(" ")],
            [inv_dvals[s] for s in desi_extra_slopes[3].split(" ")],
    ]
    desi_extra_step_funcs = [
        StepSlopes(rise=np.array(desi_extra_slopes[0], dtype=float)),
        StepSlopes(rise=np.array(desi_extra_slopes[1], dtype=float)),
        StepSlopes(rise=np.array(desi_extra_slopes[2], dtype=float)),
        StepSlopes(rise=np.array(desi_extra_slopes[3], dtype=float)),
    ]
    desi_extra_footprints = [
        Footprint(
            mjd_start,
            sun_ra_start=sun_ra_start,
            step_func=desi_extra_step_funcs[0],
            nside=nside,
        ),
        Footprint(
            mjd_start,
            sun_ra_start=sun_ra_start,
            step_func=desi_extra_step_funcs[1],
            nside=nside,
        ),
        Footprint(
            mjd_start,
            sun_ra_start=sun_ra_start,
            step_func=desi_extra_step_funcs[2],
            nside=nside,
        ),
        Footprint(
            mjd_start,
            sun_ra_start=sun_ra_start,
            step_func=desi_extra_step_funcs[3],
            nside=nside,
        )
    ]

    wfd = hp_footprints["r"] * 0
    if wfd_indx is None:
        wfd_indx = np.where(hp_footprints["r"] == 1)[0]

    wfd[wfd_indx] = 1
    non_wfd_indx = np.where(wfd == 0)[0]

    quad_splits = {}
    quad_splits[270] = slice_quad_galactic_cut(
        hp_footprints,
        wfd_indx=wfd_indx,
        ra_range=(sun_ra_start + 1.5 * np.pi, sun_ra_start + 2.0 * np.pi),
    )
    quad_splits[360] = slice_quad_galactic_cut(
        hp_footprints,
        wfd_indx=wfd_indx,
        ra_range=(sun_ra_start, sun_ra_start + np.pi / 2),
    )
    quad_splits[90] = slice_quad_galactic_cut(
        hp_footprints,
        wfd_indx=wfd_indx,
        ra_range=(sun_ra_start + np.pi / 2, sun_ra_start + np.pi),
    )
    quad_splits[180] = slice_quad_galactic_cut(
        hp_footprints,
        wfd_indx=wfd_indx,
        ra_range=(sun_ra_start + np.pi, sun_ra_start + 1.5 * np.pi),
    )
    desi_extra_splits = {}
    for i in range(4):
        desi_extra_splits[desi_extra_quads[i]] = slice_quad_galactic_cut(
            hp_footprints,
            wfd_indx=wfd_indx,
            ra_range=des_extra_ra_range[i],
            desi_y1_only=True,
            ngc=desi_extra_ngc[i],
            sgc=desi_extra_sgc[i],
        )

    for key in hp_footprints:
        temp = hp_footprints[key] + 0
        temp[wfd_indx] = 0
        fp_non_wfd.set_footprint(key, temp)

        for i in range(len(all_slopes)):
            # make a copy of the current band
            temp = hp_footprints[key] + 0
            # Set the non-rolling area to zero
            temp[non_wfd_indx] = 0

            indx = quad_splits[slice_quads[i]][slice_split_ind[i]]

            # invert the indices
            ze = temp * 0
            ze[indx] = 1
            temp = temp * ze
            rolling_footprints[i].set_footprint(key, temp)

        for i in range(4):
            # make a copy of the current band
            temp = hp_footprints[key] + 0
            # Set the non-rolling area to zero
            temp[non_wfd_indx] = 0

            indx = desi_extra_splits[desi_extra_quads[i]][1]

            # invert the indices
            ze = temp * 0
            ze[indx] = 1
            temp = temp * ze
            desi_extra_footprints[i].set_footprint(key, temp)

    result = Footprints([fp_non_wfd] + rolling_footprints + desi_extra_footprints)
    # result = Footprints([fp_non_wfd] + desi_extra_footprints)
    return result

In [310]:
def get_desi_stripes():
    # code for DESI stripes from Anand Raichoor
    names = np.array([
        "NGC-1", "NGC-2", "NGC-3", "NGC-4", "NGC-5", "NGC-6", "NGC-7", "NGC-8", "NGC-9",
        "SGC-1", "SGC-2", "SGC-3", "SGC-4", "SGC-5", "SGC-6", "SGC-7", "SGC-8", "SGC-9",
    ])
    surveys = np.array([
        "POLU+IBIS", "POLU+IBIS", "POLU+IBIS", "POLU+IBIS", "Niji+VST", "Niji+VST", "POLU+IBIS", "POLU+IBIS", "POLU+IBIS",
        "Niji+VST", "Niji+VST", "POLU+IBIS", "POLU+IBIS", "POLU+IBIS", "POLU+IBIS", "POLU+IBIS", "POLU+IBIS", "POLU+IBIS",
    ])
    # original years
    yrs = np.array([
        4, 3, 2, 2, 4, 3, 1, 1, 5,
        4, 3, 3, 2, 2, 1, 1, 4, 5,
    ])
    # swapped times for NGC "Niji+VST", "Niji+VST" fields w/ "POLU+IBIS" fields
    # reordered stripes for POU+IBIS to make bigger coherent areas
    yrs = np.array([
        4, 4, 3, 3, 2, 2, 1, 1, 5,
        4, 3, 3, 2, 2, 1, 1, 4, 5,
    ])
    ramins = np.array([
        121.01, 122.33, 123.69, 125.11, 126.60, 128.16, 129.81, 131.56, 133.43,
        315.57, 313.82, 312.19, 310.66, 309.21, 307.84, 306.53, 305.29, 304.10,
    ])
    ramaxs = np.array([
        255.71, 254.40, 253.00, 251.52, 249.93, 228.30, 225.58, 222.41, 218.59,
        44.37, 47.15, 49.56, 51.67, 53.53, 55.18, 56.64, 57.93, 59.06,
    ])
    decmins = np.array([
        11.375, 8.875, 5.875, 3.125, 0.375, -2.375, -5.375, -8.125, -10.875,
        1.625, -1.375,  -4.125, -6.875, -9.625, -12.375, -15.375, -18.125, -20.875,
    ])
    decmaxs = np.array([
        15.125, 12.375, 9.375, 6.625, 3.875, 1.125, -1.625, -4.625, -7.375,
        5.125, 2.375, -0.625, -3.375, -6.125, -8.875, -11.625, -14.625, -17.375,
    ])
    return {
        "names": names,
        "surveys": surveys,
        "yrs": yrs,
        "ramins": ramins,
        "ramaxs": ramaxs,
        "decmins": decmins,
        "decmaxs": decmaxs
    }


def plot_map_at_mjd(
    ax, footprint, mjd,
    band="i", npix=800, cmap=None,
    vmin=0, vmax=11, index_to_plot=None,
    level_factor=2, discrete=False
):
    import skyproj.hpx_utils

    imap = footprint(mjd, norm=False)[band]
    if index_to_plot is not None:
        all_index = np.arange(imap.shape[0])
        non_index_to_plot = np.setdiff1d(all_index, index_to_plot)
        imap[non_index_to_plot] = np.nan

    stdval = np.nanstd(imap)

    lon, lat, xymap = skyproj.hpx_utils.hpxmap_to_xy(
        imap,
        [0, 360],
        [-90, 90],
        xsize=npix,
    )
    vmax = int(np.ceil(vmax))

    m = ax.pcolormesh(
        lon, lat, xymap,
        cmap=cmap,
        discrete=discrete,
        vmin=vmin, vmax=vmax,
        levels=np.linspace(vmin, vmax, vmax * level_factor + 1) - 0.5 / level_factor if discrete else None,
    )

    desi = get_desi_stripes()

    cmap = uplt.Colormap('colorblind', samples=5)

    cleg = {}
    for i in range(len(desi["names"])):
        cind = int(desi["yrs"][i]) - 1
        ramin, ramax = desi["ramins"][i], desi["ramaxs"][i]
        if ramin > ramax:
            ramin = ramin - 360.0
            # ramin, ramax = ramax, ramin
        cleg[cind] = ax.plot(
            [ramin, ramax],
            [desi["decmins"][i], desi["decmins"][i]],
            color=cmap(cind % 5),
            label=f"DESI Y{cind+1}/CY{cind+1 + 27}",  # need Y1 DESI by 2028
        )
        ax.plot(
            [ramin, ramax],
            [desi["decmaxs"][i], desi["decmaxs"][i]],
            color=cmap(cind % 5),
            linestyle="dashed",
        )
        ax.plot(
            [ramin, ramin],
            [desi["decmins"][i], desi["decmaxs"][i]],
            color=cmap(cind % 5),
        )
        ax.plot(
            [ramax, ramax],
            [desi["decmins"][i], desi["decmaxs"][i]],
            color=cmap(cind % 5),
        )
    ax.invert_xaxis()
    ax.colorbar(
        m,
        ticks=np.linspace(vmin, vmax, vmax * level_factor + 1),
        minorticks=[],
        title="requested visits / avg. visits per year",
        width=0.15,
        pad=1.0,
        space=1,
    )
    ax.legend([cleg[i] for i in range(5)], loc="t", ncols=5, frameon=False)

    return stdval

In [311]:
# code from commit 5e34aa5102d3d44fc03c2a74934338fc2ba2d1b9
# on https://github.com/lsst-sims/sims_featureScheduler_runs5.3
nside = 32
survey_start_mjd = SURVEY_START_MJD

# Parameters for rolling cadence footprint definition

# Generate footprint over the sky
sky = CurrentAreaMap(nside=nside)
footprints_hp_array, labels = sky.return_maps()
# Identify pixels for rolling
roll_indx = np.where((labels == "lowdust") | (labels == "virgo"))[0]
roll_footprint = footprints_hp_array["r"] * 0
roll_footprint[roll_indx] = 1

footprints_hp = {}
for key in footprints_hp_array.dtype.names:
    footprints_hp[key] = footprints_hp_array[key]


# Define the rolling footprint
# footprints = make_rolling_footprints(
#     fp_hp=footprints_hp,
#     mjd_start=survey_start_mjd,
#     nside=nside,
#     wfd_indx=roll_indx,
#     strategy="uniform-5.3",
# )
special_footprints = make_rolling_footprints(
    fp_hp=footprints_hp,
    mjd_start=survey_start_mjd,
    nside=nside,
    wfd_indx=roll_indx,
)

slice a for quad 270: 1 1 1 U D 1 U D 1 U D 1 1 1 1 1 1
slice b for quad 270: 1 1 1 D U 1 D U 1 D U 1 1 1 1 1 1
slice a for quad 360: 1 1 U D 1 U D 1 U D 1 1 1 1 1 1 1
slice b for quad 360: 1 1 D U 1 D U 1 D U 1 1 1 1 1 1 1
slice a for quad  90: 1 1 1 U D 1 U D 1 U D 1 1 1 1 1 1
slice b for quad  90: 1 1 1 D U 1 D U 1 D U 1 1 1 1 1 1
slice a for quad 180: 1 1 1 U D 1 U D 1 U D 1 1 1 1 1 1
slice b for quad 180: 1 1 1 D U 1 D U 1 D U 1 1 1 1 1 1
north: 0 0.22975432211101002
north: 1 0.2961783439490446
north: 2 0.2697907188353048
north: 3 0.20427661510464057
north total and halves: 1.0 0.4995450409463148 0.5004549590536851
south: 0 0.06525974025974025
south: 1 0.22792207792207791
south: 2 0.43474025974025976
south: 3 0.2720779220779221
south total and halves: 1.0 0.5 0.5
north: 0 0.22975432211101002
north: 1 0.2961783439490446
north: 2 0.2697907188353048
north: 3 0.20427661510464057
north total and halves: 1.0 0.4995450409463148 0.5004549590536851
south: 0 0.06525974025974025
south: 1 0.2

In [312]:
import os
import tqdm

fdir = "wfd_uroll3_shift3_desi2r_v4_frames"
os.makedirs(fdir, exist_ok=True)
os.system(f"rm -f {fdir}/*")

years = [1, 1.2, 1.6, 1.8, 2.0, 2.2, 2.6, 2.8, 3, 3.2, 3.6, 3.8, 4]
uber_years = np.linspace(0, 4, 41)
print(uber_years)
# uber_years = [0.5, 1, 1.5, 2, 2.5]
for j, _year in tqdm.tqdm(enumerate(uber_years), total=len(uber_years)):
    years = [_year] # , 2, 3, 4, 5, 6, 7, 8, 9, 10]
    labels = [f"{(26 + y):3.1f}" for y in years]

    fig, axs = uplt.subplots(nrows=len(years), ncols=1, figsize=(10, 5 * len(years)), proj="moll", backend="cartopy")
    for i, year in enumerate(years):

        stdval = plot_map_at_mjd(
            axs[i, 0],
            special_footprints,
            survey_start_mjd + year * YEAR,
            cmap="mako_r",
            # index_to_plot=roll_indx,
            vmax=5,
            level_factor=2,
            discrete=True,
            npix=200,
        )
        # axs[i, 0].format(title=f"Y{year}/CY{labels[i]}: std. = {stdval:0.2f}", loninline=True, latlabels=True)
        axs[i, 0].format(title=f"Rubin Y{year:3.1f}/CY{labels[i]}", loninline=True, latlabels=True)

    fig.savefig(f"{fdir}/wfd_uroll3_shift3_desi2r_v4_{j:03d}.png", dpi=200)
    uplt.close(fig)


[0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1.  1.1 1.2 1.3 1.4 1.5 1.6 1.7
 1.8 1.9 2.  2.1 2.2 2.3 2.4 2.5 2.6 2.7 2.8 2.9 3.  3.1 3.2 3.3 3.4 3.5
 3.6 3.7 3.8 3.9 4. ]


100%|██████████| 41/41 [00:57<00:00,  1.40s/it]


In [313]:
import glob
import imageio.v3 as iio


def make_gif_with_imageio(frame_folder, output_path):
    # Collect and sort paths
    file_paths = sorted(glob.glob(f"{frame_folder}/*.png"))

    # Read the images into a sequence of numpy arrays
    images = [iio.imread(file_path) for file_path in file_paths]

    # Save as an animated GIF
    # Note: 'duration' here can take an explicit integer delay per frame in milliseconds
    try:
        os.remove(output_path)
    except Exception:
        pass

    iio.imwrite(output_path, images, duration=200, loop=None, optimize=True)
    print(f"GIF successfully created at: {output_path}")

# Example usage:
make_gif_with_imageio(fdir, "wfd_uroll3_shift3_desi2r_v4.gif")


GIF successfully created at: wfd_uroll3_shift3_desi2r_v4.gif
